# Grad-CAM Explanation Generation & Localization Evaluation
## Increment 2: Visual Explanations for Brain Tumour Classification

This notebook:
1. Loads the trained model
2. Generates Grad-CAM explanations
3. Evaluates localization quality with masks
4. Saves results for analysis

## Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repository (if not already done)
# !git clone <your-repo-url>
# %cd Final_year_project

In [ ]:
# Install dependencies
!pip install -q opencv-python scikit-image matplotlib seaborn tqdm

In [ ]:
# Imports
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm
import json

# Add project root and src to path
import os
if os.path.exists('/content/Final_year_project'):
    os.chdir('/content/Final_year_project')
    
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# Import modules
from src.models.classifier import TumourClassifier
from src.explainability.grad_cam import ExplanationGenerator
from src.explainability.visualization import (
    create_comparison_visualization,
    create_grid_visualization,
    overlay_heatmap
)
from src.evaluation.localization import LocalizationEvaluator

print("✓ Imports successful")
print(f"Working directory: {os.getcwd()}")

## Load Trained Model

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model - Fixed for directory format
model_path = 'models/best_model'
print(f"\nLoading model from {model_path}...")

# Load the trained model (handles PyTorch directory format)
model = torch.load(model_path, map_location=device, weights_only=False)
model.to(device)
model.eval()

print("✓ Model loaded successfully")
print(f"  Backbone: ResNet-50")
try:
    print(f"  Classes: {model.class_names}")
except AttributeError:
    print(f"  Model loaded (state dict format)")
print(f"  Device: {device}")

## Initialize Grad-CAM

In [ ]:
# Create explanation generator
explainer = ExplanationGenerator(
    model=model,
    colormap='jet',
    device=device
)

print("✓ Grad-CAM initialized")
print(f"  Target layer: {explainer.target_layer}")

## Load Dataset

In [ ]:
# Data preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

def load_image(image_path):
    """Load and preprocess image"""
    image = Image.open(image_path).convert('RGB')
    original_image = np.array(image)
    image_tensor = transform(image).unsqueeze(0)
    return image_tensor, original_image

def load_mask(mask_path, target_size=(224, 224)):
    """Load and resize mask"""
    if not os.path.exists(mask_path):
        return None
    mask = Image.open(mask_path).convert('L')
    mask = mask.resize(target_size)
    mask = np.array(mask)
    return mask

print("✓ Data utilities ready")

In [ ]:
# Get test images
data_dir = Path('data/brisc/processed/test')
class_names = ['glioma', 'meningioma', 'pituitary']

# Collect sample images from each class
sample_images = {}
for class_name in class_names:
    class_dir = data_dir / class_name
    if class_dir.exists():
        image_files = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        sample_images[class_name] = image_files[:5]  # Take first 5 from each class
        print(f"  {class_name}: {len(image_files)} images found, using {len(sample_images[class_name])} samples")

print(f"\n✓ Loaded {sum(len(v) for v in sample_images.values())} sample images")

## Test Grad-CAM on Single Image

In [ ]:
# Test on one image
test_class = 'glioma'
test_image_path = sample_images[test_class][0]

print(f"Testing Grad-CAM on: {test_image_path.name}")

# Load image
image_tensor, original_image = load_image(test_image_path)

# Make prediction
image_tensor = image_tensor.to(device)
with torch.no_grad():
    output = model(image_tensor)
    probabilities = F.softmax(output, dim=1)
    confidence, predicted_class = torch.max(probabilities, 1)

predicted_name = class_names[predicted_class.item()]
print(f"\nPrediction: {predicted_name} ({confidence.item():.2%})")
print(f"True label: {test_class}")
print(f"Correct: {predicted_name == test_class}")

# Generate Grad-CAM
cam = explainer.generate_cam(image_tensor, target_class=predicted_class.item())
print(f"\n✓ Grad-CAM generated")
print(f"  Shape: {cam.shape}")
print(f"  Min: {cam.min():.3f}, Max: {cam.max():.3f}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original
axes[0].imshow(original_image)
axes[0].set_title(f'Original\n{predicted_name} ({confidence.item():.2%})', fontweight='bold')
axes[0].axis('off')

# Heatmap
cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
    (original_image.shape[1], original_image.shape[0])
)) / 255.0
im = axes[1].imshow(cam_resized, cmap='jet', vmin=0, vmax=1)
axes[1].set_title('Grad-CAM Heatmap', fontweight='bold')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

# Overlay
overlay = overlay_heatmap(original_image, cam, alpha=0.4)
axes[2].imshow(overlay)
axes[2].set_title('Grad-CAM Overlay', fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## Generate Explanations for All Test Samples

In [ ]:
# Initialize localization evaluator
loc_evaluator = LocalizationEvaluator(binarize_strategy='otsu')

# Create output directory
output_dir = Path('results/gradcam_explanations')
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")

In [ ]:
# Process all samples
all_results = []

for class_name in class_names:
    print(f"\nProcessing {class_name}...")
    
    class_results = []
    
    for img_path in tqdm(sample_images[class_name]):
        # Load image
        image_tensor, original_image = load_image(img_path)
        
        # Make prediction
        image_tensor_device = image_tensor.to(device)
        with torch.no_grad():
            output = model(image_tensor_device)
            probabilities = F.softmax(output, dim=1)
            confidence, predicted_class = torch.max(probabilities, 1)
        
        predicted_name = class_names[predicted_class.item()]
        
        # Generate Grad-CAM
        cam = explainer.generate_cam(image_tensor_device, target_class=predicted_class.item())
        
        # Look for mask
        mask_path = img_path.parent / 'masks' / img_path.name
        mask = load_mask(mask_path) if mask_path.exists() else None
        
        # Evaluate localization if mask available
        metrics = None
        if mask is not None:
            cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
                (mask.shape[1], mask.shape[0])
            )) / 255.0
            metrics = loc_evaluator.evaluate_all(cam_resized, mask)
        
        # Create visualization
        fig = create_comparison_visualization(
            original_image=original_image,
            cam=cam,
            prediction=predicted_name,
            confidence=confidence.item(),
            probabilities=probabilities[0],
            class_names=class_names,
            mask=mask,
            metrics=metrics,
            save_path=output_dir / f"{class_name}_{img_path.stem}_gradcam.png"
        )
        plt.close(fig)
        
        # Store results
        result = {
            'image_path': str(img_path),
            'true_class': class_name,
            'predicted_class': predicted_name,
            'confidence': float(confidence.item()),
            'correct': predicted_name == class_name,
            'probabilities': probabilities[0].cpu().tolist(),
            'has_mask': mask is not None
        }
        
        if metrics:
            result['metrics'] = {k: float(v) for k, v in metrics.items()}
        
        class_results.append(result)
        all_results.append(result)
    
    print(f"  Processed {len(class_results)} images")

print(f"\n✓ Generated explanations for {len(all_results)} images")

## Analyze Results

In [ ]:
# Classification accuracy
correct = sum(1 for r in all_results if r['correct'])
total = len(all_results)

print("=" * 60)
print("CLASSIFICATION RESULTS")
print("=" * 60)
print(f"Accuracy: {correct}/{total} ({correct/total*100:.1f}%)")
print()

# Per-class accuracy
for class_name in class_names:
    class_results = [r for r in all_results if r['true_class'] == class_name]
    class_correct = sum(1 for r in class_results if r['correct'])
    print(f"  {class_name}: {class_correct}/{len(class_results)} ({class_correct/len(class_results)*100:.1f}%)")

In [ ]:
# Localization metrics (for images with masks)
results_with_masks = [r for r in all_results if r['has_mask']]

if results_with_masks:
    print("\n" + "=" * 60)
    print("LOCALIZATION METRICS (samples with masks)")
    print("=" * 60)
    print(f"Samples with masks: {len(results_with_masks)}/{total}")
    print()
    
    # Average metrics
    metric_names = ['iou', 'dice', 'pointing_game', 'energy_ratio']
    
    for metric_name in metric_names:
        values = [r['metrics'][metric_name] for r in results_with_masks]
        avg = np.mean(values)
        std = np.std(values)
        print(f"  {metric_name.upper()}: {avg:.3f} ± {std:.3f}")
    
    # Pointing game accuracy
    pg_hits = sum(1 for r in results_with_masks if r['metrics']['pointing_game'] > 0)
    print(f"\n  Pointing Game Hits: {pg_hits}/{len(results_with_masks)} ({pg_hits/len(results_with_masks)*100:.1f}%)")
else:
    print("\n⚠ No masks found for localization evaluation")

In [ ]:
# Save results to JSON
results_file = output_dir / 'gradcam_results.json'
with open(results_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n✓ Results saved to {results_file}")

## Summary Visualization

In [ ]:
# Create grid of sample explanations
sample_indices = list(range(min(12, len(all_results))))

sample_images_list = []
sample_cams = []
sample_predictions = []
sample_confidences = []
sample_titles = []

for idx in sample_indices:
    result = all_results[idx]
    img_path = Path(result['image_path'])
    
    # Load image
    image_tensor, original_image = load_image(img_path)
    
    # Generate CAM
    image_tensor = image_tensor.to(device)
    pred_idx = class_names.index(result['predicted_class'])
    cam = explainer.generate_cam(image_tensor, target_class=pred_idx)
    
    sample_images_list.append(original_image)
    sample_cams.append(cam)
    sample_predictions.append(result['predicted_class'])
    sample_confidences.append(result['confidence'])
    sample_titles.append(f"True: {result['true_class']}")

fig = create_grid_visualization(
    images=sample_images_list,
    cams=sample_cams,
    predictions=sample_predictions,
    confidences=sample_confidences,
    titles=sample_titles,
    save_path=output_dir / 'gradcam_grid_summary.png',
    ncols=4
)

plt.show()
print("✓ Summary visualization saved")

## Download Results

In [ ]:
# Zip results for download
!zip -r gradcam_results.zip results/gradcam_explanations/
print("\n✓ Results zipped")
print("  Download: gradcam_results.zip")

# In Colab, download with:
# from google.colab import files
# files.download('gradcam_results.zip')

## Conclusion

### What We Accomplished:
1. ✅ Loaded trained CNN model
2. ✅ Implemented Grad-CAM explanation generation
3. ✅ Generated visual explanations for test samples
4. ✅ Evaluated localization quality (if masks available)
5. ✅ Created comprehensive visualizations

### Next Steps:
- Review generated explanations visually
- Analyze localization metrics
- Proceed to Increment 3: Perturbation & Stability